# 13 — Unseen Motion Agent Orchestration

这个 Notebook 演示真实产品场景：用户提交项目中从未出现过的新 motion，Agent 从原始正文开始理解任务，再决定是否调用预测、检索和证据检查工具。

它与 Final Test 不同：Final Test 检查模型对未见标签的泛化能力；这里检查完整产品链路能否接收一段新的议会文本。

## 0. 流程和边界

1. 检查 motion 是否真的不在训练、验证或 Test；
2. 从正文提取政策对象、程序类型、领域和 Aye 的含义；
3. 判断文本能否安全进入预测模型；
4. 调用同一冻结架构的部署副本预测三个政党；
5. 在不知道预测方向的情况下检索历史投票、Manifesto 和 Bill 背景；
6. 证据不足时只扩大一次检索范围；
7. 生成带证据等级和限制说明的报告。

为了让近期趋势不过时，部署副本会在 Final Test 已经结束后，使用截至 2026-04-27 的全部已揭示标签重新训练。模型结构、参数和 50/50 融合公式不变，因此不会改写已公布的 Final Test 指标。

In [ ]:
# 导入本地预测、检索和可选LLM编排需要的库
import json, os, re
from getpass import getpass
from pathlib import Path

import duckdb
import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from scipy import sparse
from sentence_transformers import SentenceTransformer
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

pd.set_option('display.max_columns', 180)
pd.set_option('display.width', 180)
pd.set_option('display.max_colwidth', 180)

def find_project_dir(start):
    # 从当前目录向上查找项目根目录，兼容从仓库根目录或 Notebook 所在目录启动
    start = Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (
            (candidate / 'data' / 'raw' / 'all.csv').exists()
            and (candidate / 'processed').exists()
            and (candidate / 'notebooks').exists()
        ):
            return candidate
    raise FileNotFoundError(
        '无法定位项目根目录：请确认仓库中存在 data/raw/all.csv、processed/ 和 notebooks/。'
    )

PROJECT_DIR = find_project_dir(Path.cwd())
PROCESSED_DIR = PROJECT_DIR / 'processed'
MODEL_DIR = PROCESSED_DIR / 'model_v2'
RAG_DIR = PROCESSED_DIR / 'rag_v3'
EMBEDDING_DIR = PROCESSED_DIR / 'rag_hybrid_v7'
OUTPUT_DIR = PROCESSED_DIR / 'unseen_motion_agent_v1'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

default_parquet_path = PROJECT_DIR / 'data' / 'external' / 'divisions.parquet'
download_parquet_path = Path.home() / 'Downloads' / 'divisions.parquet'
DIVISIONS_PARQUET_PATH = Path(os.environ.get(
    'DIVISIONS_PARQUET_PATH',
    default_parquet_path if default_parquet_path.exists() else download_parquet_path,
)).expanduser().resolve()
RAW_ALL_PATH = PROJECT_DIR / 'data' / 'raw' / 'all.csv'
DATA_CUTOFF = pd.Timestamp('2026-04-27')
MAX_CHALLENGE_GAP_DAYS = 45
IN_SCOPE_PARTIES = ['labour', 'conservative', 'liberal-democrat']
PARTY_ROLES = {'labour': 'governing_party', 'conservative': 'main_opposition', 'liberal-democrat': 'smaller_opposition'}

# 第一次运行保持False；免费阶段会完成预测、检索和工具轨迹
RUN_LLM_PLANNER = False
RUN_LLM_REPORTS = False
LLM_MODEL = 'gpt-4o-mini-2024-07-18'
LLM_HARD_BUDGET_USD = 0.20

print('Notebook version: 13-unseen-motion-agent-orchestration-v1')
print('Data cutoff:', DATA_CUTOFF.date())
print('LLM planner:', RUN_LLM_PLANNER, '| LLM reports:', RUN_LLM_REPORTS)
print('Output directory:', OUTPUT_DIR)

## 1. 检查外部 division 文件并锁定案例

`divisions.parquet` 共有 division 元数据、日期、标题、`motion_id` 和辩论标识，但没有完整 `motion_text`。它可以证明案例是新的；用于解析的正文来自每条记录对应的公开 TheyWorkForYou Votes 页面。

案例日期被限制在原数据截止日之后45天内，避免把“处理新输入”和“政治趋势已过期”混在一起。

In [ ]:
# 使用DuckDB读取Parquet，不需要额外安装pyarrow
required = [DIVISIONS_PARQUET_PATH, RAW_ALL_PATH]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError('缺少必要文件：' + ' | '.join(missing))

connection = duckdb.connect()
divisions = connection.execute('SELECT * FROM read_parquet(?)', [str(DIVISIONS_PARQUET_PATH)]).df()
existing_keys = set(pd.read_csv(RAW_ALL_PATH, usecols=['division_key'])['division_key'].dropna().astype(str))
divisions['is_in_original_project'] = divisions['key'].astype(str).isin(existing_keys)
divisions['date'] = pd.to_datetime(divisions['date'])
new_commons = divisions.loc[divisions['chamber_slug'].eq('commons') & ~divisions['is_in_original_project']].copy()
post_cutoff = new_commons.loc[new_commons['date'].gt(DATA_CUTOFF)].copy()

division_audit = pd.DataFrame([{
    'parquet_rows': len(divisions),
    'parquet_commons_divisions': int(divisions['chamber_slug'].eq('commons').sum()),
    'overlap_with_project_commons': int((divisions['chamber_slug'].eq('commons') & divisions['is_in_original_project']).sum()),
    'new_commons_divisions': len(new_commons),
    'new_commons_after_cutoff': len(post_cutoff),
    'parquet_has_motion_text': 'motion_text' in divisions.columns,
}])
display(division_audit)
division_audit.to_csv(OUTPUT_DIR / 'external_division_audit_v1.csv', index=False)

# 正文来自公开页面；这里只保存motion，不读取四个案例的真实投票结果
UNSEEN_CASES = [
 {
  'case_id':'U01_ASYLUM_RECEPTION_SI','division_key':'pw-2026-04-28-514-commons','motion_date':'2026-04-28',
  'motion_title':'Draft Asylum Seekers (Reception Conditions) (Amendment) Regulations 2026',
  'legislation_name':'Asylum Seekers (Reception Conditions) (Amendment) Regulations 2026',
  'motion_text':'Motion made, and Question put forthwith (Standing Order No. 118(6)), That the draft Asylum Seekers (Reception Conditions) (Amendment) Regulations 2026, which were laid before this House on 5 March, be approved.—(Taiwo Owatemi.)',
  'total_possible_members':650,'source_url':'https://votes.theyworkforyou.com/decisions/division/commons/2026-04-28/514'
 },
 {
  'case_id':'U02_KINGS_SPEECH_MULTI_ISSUE','division_key':'pw-2026-05-20-2-commons','motion_date':'2026-05-20',
  'motion_title':"King's Speech Motion for an Address: amendment (o)",'legislation_name':'',
  'motion_text':'I beg to move amendment (o), at the end of the Question to add: “but respectfully regret the absence of a Defence Readiness Bill from the Gracious Speech, and the 10 month delay to the publication of the Defence Investment Plan; call on the Government to bring forward both as a matter of urgency; further call on the Government to raise defence spending to 3% of GDP by the end of this Parliament; further regret that the Gracious Speech commits to re-starting inquests into Northern Ireland veterans; and also call on the Government not to progress with the Northern Ireland Troubles Bill and the Northern Ireland Troubles (Legacy and Reconciliation) Act 2023 (Remedial) Order 2025 to protect veterans and improve the morale of all who serve in the armed forces.”',
  'total_possible_members':647,'source_url':'https://votes.theyworkforyou.com/decisions/division/commons/2026-05-20/2'
 },
 {
  'case_id':'U03_STEEL_REASONED_AMENDMENT','division_key':'pw-2026-05-21-6-commons','motion_date':'2026-05-21',
  'motion_title':'Steel Industry (Nationalisation) Bill: Reasoned Amendment to Second Reading','legislation_name':'Steel Industry (Nationalisation) Bill',
  'motion_text':'I beg to move an amendment, to leave out from “That” to the end of the Question and add: “this House declines to give a Second Reading to the Steel Industry (Nationalisation) Bill because it believes that politicians should not be running businesses; because expropriating businesses sets a precedent that will deter inward investment into other UK businesses; because the Bill exposes taxpayers to unlimited liabilities; because the powers that the Bill confers on Ministers are far wider in scope than would be required for its stated purpose; and because it fails to contain any measures that would address the issues which are currently making domestic production of steel unprofitable such as higher employment costs and policies in pursuit of net zero, such as carbon taxes and associated regulations and levies.”',
  'total_possible_members':647,'source_url':'https://votes.theyworkforyou.com/decisions/division/commons/2026-05-21/6'
 },
 {
  'case_id':'U04_ARMED_FORCES_ILR_FEES','division_key':'pw-2026-06-02-8-commons','motion_date':'2026-06-02',
  'motion_title':'Armed Forces Bill Committee: New Clause 5','legislation_name':'Armed Forces Bill',
  'motion_text':'New Clause 5 - Waived fees for indefinite leave to remain for spouses or dependants of serving or discharged member of the armed forces. (1) The Immigration Act 2014 is amended as follows. (2) In section 68, after subsection (11) insert—“11A Fees may not be charged. No fees may be charged in respect of a serving or previously serving member of the armed forces or their family members applying for indefinite leave to remain under the Immigration Rules Appendix HM Armed Forces.” —(Mr Francois.)',
  'total_possible_members':647,'source_url':'https://votes.theyworkforyou.com/decisions/division/commons/2026-06-02/8'
 }
]
case_registry = pd.DataFrame(UNSEEN_CASES)
case_registry['motion_date'] = pd.to_datetime(case_registry['motion_date'])
case_registry['days_after_cutoff'] = (case_registry['motion_date'] - DATA_CUTOFF).dt.days
case_registry['absent_from_original_project'] = ~case_registry['division_key'].isin(existing_keys)
case_registry['present_in_external_parquet'] = case_registry['division_key'].isin(set(divisions['key'].astype(str)))
assert len(case_registry) == 4 and case_registry['division_key'].is_unique
assert case_registry['absent_from_original_project'].all() and case_registry['present_in_external_parquet'].all()
assert case_registry['days_after_cutoff'].between(1, MAX_CHALLENGE_GAP_DAYS).all()
case_registry.to_csv(OUTPUT_DIR / 'unseen_case_registry_v1.csv', index=False)
display(case_registry[['case_id','motion_date','days_after_cutoff','motion_title','source_url']])

## 2. Agent规划与政策对象合同

免费模式使用已经逐字核对的合同。开启 `RUN_LLM_PLANNER` 后，LLM会从相同原文提取任务合同并提出工具需求；控制器再依据是否允许预测生成最终工具清单，并检查枚举、原文引文和多议题安全门。

LLM只负责理解任务。概率由模型计算，证据等级由代码判断。

In [ ]:
# 免费合同同时充当LLM规划结果的人工审计基准
REFERENCE_CONTRACTS = {
 'U01_ASYLUM_RECEPTION_SI': dict(prediction_eligible=True,ineligibility_reason='',policy_object='approve the 2026 amendments to reception conditions for asylum seekers',motion_type='approve_statutory_instrument',motion_family='delegated_legislation',policy_domain_primary='immigration_asylum',policy_action='approve_regulations',aye_effect_on_object=1,issue_count_estimate=1,object_government_backed=1,government_backing_known=1,source_quote='That the draft Asylum Seekers (Reception Conditions) (Amendment) Regulations 2026, which were laid before this House on 5 March, be approved',extraction_confidence='high'),
 'U02_KINGS_SPEECH_MULTI_ISSUE': dict(prediction_eligible=False,ineligibility_reason='同一修正案包含多项可独立判断的政策要求，单一概率会掩盖议题冲突。',policy_object='a multi-issue amendment covering defence spending, legislation, Northern Ireland inquests and veterans policy',motion_type='amendment',motion_family='amendment_or_clause',policy_domain_primary='defence_foreign_affairs',policy_action='add_multi_issue_criticism_and_demands',aye_effect_on_object=1,issue_count_estimate=5,object_government_backed=0,government_backing_known=1,source_quote='call on the Government to raise defence spending to 3% of GDP by the end of this Parliament',extraction_confidence='low'),
 'U03_STEEL_REASONED_AMENDMENT': dict(prediction_eligible=True,ineligibility_reason='',policy_object='give the Steel Industry (Nationalisation) Bill a Second Reading and permit steel nationalisation',motion_type='reasoned_amendment',motion_family='amendment_or_clause',policy_domain_primary='economy_finance_business',policy_action='decline_second_reading',aye_effect_on_object=-1,issue_count_estimate=1,object_government_backed=1,government_backing_known=1,source_quote='this House declines to give a Second Reading to the Steel Industry (Nationalisation) Bill',extraction_confidence='high'),
 'U04_ARMED_FORCES_ILR_FEES': dict(prediction_eligible=True,ineligibility_reason='',policy_object='waive indefinite-leave-to-remain fees for armed forces members and their families',motion_type='proposed_clause',motion_family='amendment_or_clause',policy_domain_primary='defence_foreign_affairs',policy_action='waive_immigration_fees',aye_effect_on_object=1,issue_count_estimate=1,object_government_backed=0,government_backing_known=1,source_quote='No fees may be charged in respect of a serving or previously serving member of the armed forces or their family members applying for indefinite leave to remain',extraction_confidence='high')
}
ALLOWED_TOOLS = {'predict_stance','retrieve_evidence','assess_evidence','generate_report'}
ALLOWED_TYPES = {'approve_statutory_instrument','amendment','reasoned_amendment','proposed_clause','second_stage','third_stage','other'}
ALLOWED_FAMILIES = {'delegated_legislation','amendment_or_clause','bill_stage','other_motion'}
ALLOWED_DOMAINS = {'economy_finance_business','constitution_democracy_devolution','employment_labour','transport_infrastructure','education_skills_children','health_social_care','housing_planning_local_government','welfare_pensions_poverty','agriculture_food_rural','defence_foreign_affairs','digital_data_media_ai','environment_energy_water','immigration_asylum','justice_crime_policing','other_or_unclear'}

def normalize_text(value):
    # 统一弯引号和空白，只用于检查引文是否来自原文
    value = str(value).lower().replace('“','').replace('”','').replace('’',"'")
    return re.sub(r'\s+',' ',value).strip(' .—–-')

def enforce_controller_tool_plan(contract):
    # 工具调用由控制器统一编排，避免LLM的判断与工具清单偶尔互相矛盾
    result = dict(contract)
    result['requested_parties'] = list(IN_SCOPE_PARTIES)
    result['tools_needed'] = ['retrieve_evidence','assess_evidence','generate_report']
    if result['prediction_eligible']:
        result['tools_needed'].insert(0,'predict_stance')
    return result

def with_tool_plan(contract):
    # 免费人工合同和LLM合同使用同一套控制器规则
    return enforce_controller_tool_plan(contract)

def reconcile_planner_contract(case, contract):
    # LLM可以做语义理解，但程序负责把输出变成稳定、可执行且可审计的合同
    result = dict(contract)
    adjustments = []

    # Second Reading、法定文书和reasoned amendment虽然可能包含多项理由，
    # 最终表决动作仍是一个：通过、否决或拒绝进入下一阶段。
    single_decision_types = {
        'approve_statutory_instrument', 'second_stage', 'third_stage',
        'reasoned_amendment', 'proposed_clause'
    }
    if result.get('motion_type') in single_decision_types and int(result.get('issue_count_estimate', 1)) != 1:
        result['issue_count_estimate'] = 1
        adjustments.append('issue_count_normalized_to_single_decision')

    # 如果LLM把一个单一表决误写成“多议题不可预测”，纠正为可预测。
    reason = str(result.get('ineligibility_reason', ''))
    if (
        not bool(result.get('prediction_eligible'))
        and int(result.get('issue_count_estimate', 1)) < 3
        and '多议题' in reason
        and str(result.get('policy_object', '')).strip()
    ):
        result['prediction_eligible'] = True
        result['ineligibility_reason'] = ''
        adjustments.append('false_multi_issue_block_removed')

    # 引文必须是motion原文。LLM若做了轻微改写，优先换回已审核的逐字引文。
    if normalize_text(result.get('source_quote', '')) not in normalize_text(case['motion_text']):
        reviewed_quote = str(REFERENCE_CONTRACTS.get(case['case_id'], {}).get('source_quote', ''))
        if reviewed_quote and normalize_text(reviewed_quote) in normalize_text(case['motion_text']):
            result['source_quote'] = reviewed_quote
            adjustments.append('source_quote_reanchored_to_reviewed_literal')
        else:
            # 没有可靠逐字引文时保留原文片段，但关闭预测，避免把改写内容当成证据。
            result['source_quote'] = str(case['motion_text'])[:400].strip()
            result['prediction_eligible'] = False
            result['ineligibility_reason'] = '规划器引文无法定位到原文，需要人工确认政策对象。'
            result['extraction_confidence'] = 'low'
            adjustments.append('prediction_blocked_unanchored_quote')

    result = enforce_controller_tool_plan(result)
    return result, adjustments

def validate_contract(case, contract):
    return {
      'tools_allowed': set(contract['tools_needed']).issubset(ALLOWED_TOOLS),
      'parties_allowed': set(contract['requested_parties']).issubset(IN_SCOPE_PARTIES),
      'motion_type_allowed': contract['motion_type'] in ALLOWED_TYPES,
      'motion_family_allowed': contract['motion_family'] in ALLOWED_FAMILIES,
      'domain_allowed': contract['policy_domain_primary'] in ALLOWED_DOMAINS,
      'aye_effect_allowed': contract['aye_effect_on_object'] in {-1,1},
      'source_quote_anchored': normalize_text(contract['source_quote']) in normalize_text(case['motion_text']),
      'multi_issue_not_forced': not (int(contract['issue_count_estimate']) >= 3 and bool(contract['prediction_eligible'])),
      'prediction_tool_consistent': (('predict_stance' in contract['tools_needed']) == bool(contract['prediction_eligible']))
    }

# 可选LLM规划器：每个案例一次调用，严格输出JSON；默认不付费
PLANNER_SCHEMA = {
 'type':'object','additionalProperties':False,
 'properties':{
  'prediction_eligible':{'type':'boolean'},'ineligibility_reason':{'type':'string'},'policy_object':{'type':'string'},
  'motion_type':{'type':'string','enum':sorted(ALLOWED_TYPES)},'motion_family':{'type':'string','enum':sorted(ALLOWED_FAMILIES)},
  'policy_domain_primary':{'type':'string','enum':sorted(ALLOWED_DOMAINS)},'policy_action':{'type':'string'},
  'aye_effect_on_object':{'type':'integer','enum':[-1,1]},'issue_count_estimate':{'type':'integer','minimum':1},
  'object_government_backed':{'type':'integer','enum':[0,1]},'government_backing_known':{'type':'integer','enum':[0,1]},
  'source_quote':{'type':'string'},'extraction_confidence':{'type':'string','enum':['high','medium','low']},
  'requested_parties':{'type':'array','items':{'type':'string','enum':IN_SCOPE_PARTIES}},
  'tools_needed':{'type':'array','items':{'type':'string','enum':sorted(ALLOWED_TOOLS)}}
 },
 'required':['prediction_eligible','ineligibility_reason','policy_object','motion_type','motion_family','policy_domain_primary','policy_action','aye_effect_on_object','issue_count_estimate','object_government_backed','government_backing_known','source_quote','extraction_confidence','requested_parties','tools_needed']
}
planner_cache_path = OUTPUT_DIR / 'llm_planner_contracts_v1.jsonl'
planner_cache = {}
if planner_cache_path.exists():
    for line in planner_cache_path.read_text(encoding='utf-8').splitlines():
        if line.strip():
            record = json.loads(line); planner_cache[record['case_id']] = record

if RUN_LLM_PLANNER:
    from openai import OpenAI
    key = os.environ.get('OPENAI_API_KEY','').strip() or getpass('请粘贴OpenAI API key：').strip()
    if not key: raise EnvironmentError('没有提供API key。')
    planner_client = OpenAI(api_key=key)
    for case in UNSEEN_CASES:
        if case['case_id'] in planner_cache: continue
        prompt = json.dumps({'user_request':'分析这项法案，各党可能持什么立场？证据是否充分？','motion_title':case['motion_title'],'motion_text':case['motion_text'],'allowed_parties':IN_SCOPE_PARTIES},ensure_ascii=False)
        response = planner_client.responses.create(model=LLM_MODEL,instructions='你是受控Agent规划器。只从原文抽取，不预测立场。多议题且不能归一为单一政策对象时必须阻止预测，但仍可检索和生成限制说明。',input=prompt,text={'format':{'type':'json_schema','name':'unseen_motion_plan','strict':True,'schema':PLANNER_SCHEMA}},temperature=0,max_output_tokens=900,store=False)
        payload = json.loads(response.output_text)
        llm_requested_parties = list(payload.get('requested_parties', []))
        llm_tools_needed = list(payload.get('tools_needed', []))
        original_payload = dict(payload)
        payload, controller_adjustments = reconcile_planner_contract(case, payload)
        checks = validate_contract(case,payload)
        if not all(checks.values()): raise ValueError(f"{case['case_id']}规划失败：{checks}")
        record={'case_id':case['case_id'],'contract':payload,'checks':checks,'raw_llm_contract':original_payload,'llm_requested_parties':llm_requested_parties,'llm_tools_needed':llm_tools_needed,'controller_adjustments':controller_adjustments,'controller_tool_plan_applied':llm_requested_parties != payload['requested_parties'] or llm_tools_needed != payload['tools_needed']}
        with planner_cache_path.open('a',encoding='utf-8') as handle: handle.write(json.dumps(record,ensure_ascii=False)+'\n')
        planner_cache[case['case_id']]=record

contracts_by_case = {}
rows=[]
for case in UNSEEN_CASES:
    contract = planner_cache.get(case['case_id'],{}).get('contract') or REFERENCE_CONTRACTS[case['case_id']]
    contract, controller_adjustments = reconcile_planner_contract(case, contract)
    checks = validate_contract(case,contract)
    if not all(checks.values()): raise ValueError(f"{case['case_id']}合同失败：{checks}")
    contracts_by_case[case['case_id']] = contract
    rows.append({'case_id':case['case_id'],**contract,'planner_source':'llm' if case['case_id'] in planner_cache else 'reviewed_free_preview','controller_adjustments':' | '.join(controller_adjustments)})
contracts = pd.DataFrame(rows)
contracts.to_csv(OUTPUT_DIR/'unseen_case_contracts_v1.csv',index=False)
display(contracts[['case_id','planner_source','prediction_eligible','policy_object','motion_type','policy_domain_primary','tools_needed']])

## 3. 刷新部署模型并预测

Final Test 指标保持冻结。产品部署副本使用相同架构和参数，把截至2026-04-27已经揭示的Test历史加入训练，并重新计算每个政党最近100次投票先验。四个挑战案例的投票结果不会被读取。

In [ ]:
TARGET='target_binary_support'
ROLE_CATEGORICAL=['party_role','final_object_government_backed','government_backing_known','motion_type','motion_family','policy_domain_primary','role_backing_interaction']
ROLE_NUMERIC=['motion_month','total_possible_members','is_governing_party','is_main_opposition','is_opposition_day','is_amendment','is_new_clause','is_financial_motion','policy_domain_count']

def add_role_features(frame):
    # 与09B和10保持完全相同的输入结构
    result=frame.copy(); result['motion_date']=pd.to_datetime(result['motion_date'])
    result['role_backing_interaction']=result['party_role'].fillna('unknown').astype(str)+'__'+result['final_object_government_backed'].fillna('unknown').astype(str)
    for column in ROLE_CATEGORICAL: result[column]=result[column].fillna('__missing__').astype(str)
    for column in ROLE_NUMERIC: result[column]=pd.to_numeric(result[column],errors='coerce')
    return result

def make_preprocessor():
    cat=Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('one_hot',OneHotEncoder(handle_unknown='ignore'))])
    num=Pipeline([('imputer',SimpleImputer(strategy='median')),('scale',StandardScaler(with_mean=False))])
    return ColumnTransformer([('categorical',cat,ROLE_CATEGORICAL),('numeric',num,ROLE_NUMERIC)],remainder='drop')

def make_model():
    # 参数在新案例出现前已经冻结
    return XGBClassifier(n_estimators=300,max_depth=3,learning_rate=0.03,min_child_weight=10,subsample=0.8,colsample_bytree=0.8,reg_lambda=5.0,objective='binary:logistic',eval_metric='logloss',tree_method='hist',random_state=42,n_jobs=-1)

def decay_weights(frame,reference):
    age=(reference-frame['motion_date']).dt.days.clip(lower=0)/365.25
    return np.power(0.5,age/2.0)

parts=[]
for name in ['model_train_v2.csv','model_validation_v2.csv','model_test_v2.csv']:
    part=pd.read_csv(MODEL_DIR/name,low_memory=False); part['motion_date']=pd.to_datetime(part['motion_date']); parts.append(part)
history=pd.concat(parts,ignore_index=True)
history=history.loc[history['party'].isin(IN_SCOPE_PARTIES)&history['motion_date'].le(DATA_CUTOFF)&history[TARGET].notna()].copy()
history[TARGET]=pd.to_numeric(history[TARGET],errors='raise').astype(int)
assert history['motion_date'].max()<=DATA_CUTOFF
assert not set(case_registry['division_key'])&set(history['division_key'])
history=add_role_features(history)
preprocessor=make_preprocessor(); history_matrix=preprocessor.fit_transform(history)
role_model=make_model(); role_model.fit(history_matrix,history[TARGET],sample_weight=decay_weights(history,DATA_CUTOFF+pd.Timedelta(days=1)))
recent=history.sort_values('motion_date').groupby('party',group_keys=False).tail(100)
prior_rates={}
for party in IN_SCOPE_PARTIES:
    party_recent=recent.loc[recent['party'].eq(party)]
    if len(party_recent)!=100: raise ValueError(f'{party}不足100条近期投票。')
    prior_rates[party]=float((party_recent[TARGET].sum()+1)/(len(party_recent)+2))

feature_rows=[]
for _,case in case_registry.iterrows():
    contract=contracts_by_case[case['case_id']]
    if not contract['prediction_eligible']: continue
    for party in IN_SCOPE_PARTIES:
        role=PARTY_ROLES[party]; mtype=contract['motion_type']
        feature_rows.append({'row_id':f"{case['division_key']}__{party}",'case_id':case['case_id'],'division_key':case['division_key'],'motion_date':case['motion_date'],'party':party,'party_role':role,'motion_title_clean':case['motion_title'],'final_policy_object':contract['policy_object'],'final_object_government_backed':float(contract['object_government_backed']),'government_backing_known':int(contract['government_backing_known']),'motion_type':mtype,'motion_family':contract['motion_family'],'policy_domain_primary':contract['policy_domain_primary'],'motion_month':case['motion_date'].month,'total_possible_members':case['total_possible_members'],'is_governing_party':int(role=='governing_party'),'is_main_opposition':int(role=='main_opposition'),'is_opposition_day':int('opposition day' in case['motion_title'].lower()),'is_amendment':int(mtype in {'amendment','reasoned_amendment'}),'is_new_clause':int(mtype=='proposed_clause'),'is_financial_motion':0,'policy_domain_count':contract['issue_count_estimate'],'aye_effect_on_object':contract['aye_effect_on_object']})
features=add_role_features(pd.DataFrame(feature_rows))
role_p=role_model.predict_proba(preprocessor.transform(features))[:,1]
prior_p=features['party'].map(prior_rates).astype(float).to_numpy()
support_p=0.5*role_p+0.5*prior_p
predictions=features[['row_id','case_id','division_key','motion_date','party','party_role','motion_title_clean','final_policy_object','motion_type','motion_family','policy_domain_primary','aye_effect_on_object']].copy()
predictions['role_probability']=role_p; predictions['recent_prior_probability']=prior_p
predictions['support_probability_policy_object']=support_p
predictions['predicted_policy_stance']=np.where(support_p>=0.5,'support','oppose')
predictions['aye_probability']=np.where(predictions['aye_effect_on_object'].eq(1),support_p,1-support_p)
predictions['raw_vote_prediction']=np.where(predictions['aye_probability']>=0.5,'Aye','No')
predictions['challenge_outcome_used']=False
predictions.to_csv(OUTPUT_DIR/'unseen_party_predictions_v1.csv',index=False)
manifest={'history_rows':len(history),'history_divisions':history['division_key'].nunique(),'history_max_date':str(history['motion_date'].max().date()),'model_design':'role_prior_blend_50_50','parameters_changed':False,'final_test_metrics_changed':False,'challenge_outcomes_read':False,'recent_prior_rates':prior_rates}
(OUTPUT_DIR/'deployment_refresh_manifest_v1.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
display(predictions[['case_id','party','predicted_policy_stance','support_probability_policy_object','raw_vote_prediction','aye_probability']].round(4))

## 4. Hybrid RAG和一次补充检索

查询不包含预测方向。第一轮使用较高相关性门槛；若少于两条，第二轮只放宽到同领域或相近对象。第二轮材料不能升级为 Direct。

In [ ]:
chunks=pd.read_csv(RAG_DIR/'rag_chunks_v3.csv',low_memory=False); chunks['source_date']=pd.to_datetime(chunks['source_date'],errors='coerce')
tfidf_matrix=sparse.load_npz(RAG_DIR/'sparse_tfidf_matrix_v3.npz')
vectorizer=joblib.load(RAG_DIR/'sparse_tfidf_vectorizer_v3.joblib')
chunk_embeddings=np.load(EMBEDDING_DIR/'chunk_embeddings_v7.npy')
assert len(chunks)==tfidf_matrix.shape[0]==chunk_embeddings.shape[0] and chunk_embeddings.shape[1]==384
embedding_model=SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
STOP={'that','this','with','from','into','under','have','been','will','would','should','could','their','there','which','house','bill','motion','clause','amendment','section','regulations','government'}

def tokens(text): return {word for word in re.findall(r"[a-z][a-z'-]{3,}",str(text).lower()) if word not in STOP}
def overlap(a,b):
    left=tokens(a); return len(left&tokens(b))/len(left) if left else 0.0

def retrieve(case,contract,party,max_results=6):
    # 搜索文本不出现support、oppose或模型概率
    query=f"Motion title: {case['motion_title']}\nPolicy object: {contract['policy_object']}\nMotion text: {case['motion_text']}"
    sparse_q=vectorizer.transform([query]); tfidf=(tfidf_matrix@sparse_q.T).toarray().ravel()
    dense_q=embedding_model.encode([query],normalize_embeddings=True,convert_to_numpy=True,show_progress_bar=False).astype('float32')[0]
    frame=chunks.copy(); frame['tfidf_score']=tfidf; frame['dense_score']=chunk_embeddings@dense_q; frame['hybrid_score']=0.5*frame['tfidf_score']+0.5*frame['dense_score']
    frame=frame.loc[frame['source_date'].lt(case['motion_date'])&frame['division_key'].fillna('').ne(case['division_key'])&(frame['party'].fillna('').eq(party)|frame['source_type'].eq('bill_reference'))].copy()
    frame['lexical_overlap']=frame['retrieval_text'].map(lambda value:overlap(contract['policy_object'],value))
    frame['same_domain']=frame['policy_domain'].fillna('').eq(contract['policy_domain_primary'])
    first_pool=frame.loc[frame['hybrid_score'].ge(0.25)&(frame['same_domain']|frame['lexical_overlap'].ge(0.20))].sort_values('hybrid_score',ascending=False).copy()
    strong=((first_pool['source_type'].eq('historical_vote')&first_pool['lexical_overlap'].ge(0.60)&first_pool['hybrid_score'].ge(0.35))|(first_pool['source_type'].eq('manifesto')&first_pool['lexical_overlap'].ge(0.25)&first_pool['hybrid_score'].ge(0.25)))
    retry=int(strong.sum())==0
    first=first_pool.head(3 if retry else max_results).copy(); first['retrieval_round']=1; selected=first
    if retry:
        second=frame.loc[~frame['chunk_id'].isin(set(first['chunk_id']))&frame['hybrid_score'].ge(0.16)&(frame['same_domain']|frame['lexical_overlap'].ge(0.08))].sort_values('hybrid_score',ascending=False).head(max_results-len(first)).copy(); second['retrieval_round']=2
        selected=pd.concat([first,second],ignore_index=True)
    selected=selected.sort_values('hybrid_score',ascending=False).drop_duplicates('document_id').head(max_results).copy()
    if selected.empty: return selected,retry
    def role(row):
        if row['retrieval_round']==2:
            return {'historical_vote':'similar_historical_vote','manifesto':'related_manifesto_context'}.get(row['source_type'],'bill_background')
        if row['source_type']=='historical_vote' and row['lexical_overlap']>=0.60 and row['hybrid_score']>=0.35: return 'parliamentary_direct_candidate'
        if row['source_type']=='manifesto' and row['lexical_overlap']>=0.25 and row['hybrid_score']>=0.25: return 'official_policy_direction_candidate'
        return {'historical_vote':'similar_historical_vote','manifesto':'related_manifesto_context'}.get(row['source_type'],'bill_background')
    selected['evidence_role']=selected.apply(role,axis=1); selected['case_id']=case['case_id']; selected['query_party']=party; selected['query_date']=case['motion_date']; selected['query_division_key']=case['division_key']; selected['retrieval_used_predicted_stance']=False; selected['rank']=range(1,len(selected)+1); selected['evidence_id']=[f'E{i}' for i in range(1,len(selected)+1)]
    return selected,retry

evidence_parts=[]; status_rows=[]
for _,case_row in case_registry.iterrows():
    case=case_row.to_dict(); contract=contracts_by_case[case['case_id']]
    for party in IN_SCOPE_PARTIES:
        found,retry=retrieve(case,contract,party)
        if not found.empty: evidence_parts.append(found); roles=set(found['evidence_role'])
        else: roles=set()
        status='direct_candidate_requires_review' if 'parliamentary_direct_candidate' in roles else ('official_policy_evidence' if 'official_policy_direction_candidate' in roles else ('related_context_only' if roles else 'insufficient_evidence'))
        status_rows.append({'query_id':f"{case['case_id']}__{party}",'case_id':case['case_id'],'party':party,'prediction_eligible':contract['prediction_eligible'],'retry_used':retry,'evidence_count':len(found),'evidence_status':status})
evidence=pd.concat(evidence_parts,ignore_index=True) if evidence_parts else pd.DataFrame()
query_status=pd.DataFrame(status_rows)
if not evidence.empty: evidence.to_csv(OUTPUT_DIR/'unseen_retrieval_evidence_v1.csv',index=False)
query_status.to_csv(OUTPUT_DIR/'unseen_query_evidence_status_v1.csv',index=False)
display(query_status)
if not evidence.empty: display(evidence[['case_id','query_party','rank','retrieval_round','source_type','evidence_role','title','hybrid_score','lexical_overlap']].round(4))

## 5. Agent工具轨迹与可选最终报告

Agent能力以可检查的工具决策呈现。开启 `RUN_LLM_REPORTS` 后每个案例只调用一次LLM；LLM只能写解释，不能修改概率和证据状态。

In [ ]:
trace=[]
for _,case in case_registry.iterrows():
    contract=contracts_by_case[case['case_id']]; status=query_status.loc[query_status['case_id'].eq(case['case_id'])]
    trace += [
      {'case_id':case['case_id'],'step':1,'tool':'understand_task','decision':'prediction_eligible' if contract['prediction_eligible'] else 'prediction_blocked','detail':contract['ineligibility_reason'] or contract['policy_object']},
      {'case_id':case['case_id'],'step':2,'tool':'predict_stance','decision':'called' if contract['prediction_eligible'] else 'skipped','detail':'三个政党分别预测；多议题不强行输出单一概率。'},
      {'case_id':case['case_id'],'step':3,'tool':'retrieve_evidence','decision':'retry_once' if status['retry_used'].any() else 'single_pass','detail':'查询未包含预测方向。'},
      {'case_id':case['case_id'],'step':4,'tool':'assess_evidence','decision':' | '.join(sorted(status['evidence_status'].unique())),'detail':f"返回{int(status['evidence_count'].sum())}条材料。"},
      {'case_id':case['case_id'],'step':5,'tool':'generate_report','decision':'ready_with_limits','detail':'RAG不能覆盖模型概率。'}
    ]
agent_trace=pd.DataFrame(trace); agent_trace.to_csv(OUTPUT_DIR/'agent_tool_trace_v1.csv',index=False); display(agent_trace)

def build_prompt(case_id):
    case=case_registry.set_index('case_id').loc[case_id]; contract=contracts_by_case[case_id]
    pred=predictions.loc[predictions['case_id'].eq(case_id),['party','predicted_policy_stance','support_probability_policy_object','raw_vote_prediction','aye_probability']].round(4).to_dict('records')
    ev=[]
    if not evidence.empty:
        for _,row in evidence.loc[evidence['case_id'].eq(case_id)].iterrows(): ev.append({'evidence_id':f"{row['query_party']}:{row['evidence_id']}",'party':row['query_party'],'source_type':row['source_type'],'evidence_role':row['evidence_role'],'title':row['title'],'date':str(pd.Timestamp(row['source_date']).date()),'stance_label':None if pd.isna(row['stance_label']) else row['stance_label'],'excerpt':str(row['evidence_text'])[:600]})
    payload={'request':'分析这项法案，各党可能持什么立场？证据是否充分？','motion':{'case_id':case_id,'title':case['motion_title'],'date':str(case['motion_date'].date()),'text':case['motion_text']},'validated_contract':contract,'frozen_predictions':pred,'evidence_status':query_status.loc[query_status['case_id'].eq(case_id)].to_dict('records'),'retrieved_material':ev,'rules':['不得修改或补造概率','不得把相关背景称为直接证据','prediction_eligible为false时解释拒绝单一概率','分别说明模型判断、证据和不确定性']}
    return json.dumps(payload,ensure_ascii=False,indent=2)

cache_path=OUTPUT_DIR/'unseen_agent_llm_reports_v1.jsonl'; cache={}
if cache_path.exists():
    for line in cache_path.read_text(encoding='utf-8').splitlines():
        if line.strip(): record=json.loads(line); cache[record['case_id']]=record
if RUN_LLM_REPORTS:
    from openai import OpenAI
    key=os.environ.get('OPENAI_API_KEY','').strip() or getpass('请粘贴OpenAI API key：').strip()
    if not key: raise EnvironmentError('没有提供API key。')
    client=OpenAI(api_key=key); cost=sum(float(item.get('actual_cost_usd',0)) for item in cache.values())
    for case_id in case_registry['case_id']:
        if case_id in cache: continue
        if cost>=LLM_HARD_BUDGET_USD: print('已触发0.20美元预算保护。'); break
        response=client.responses.create(model=LLM_MODEL,instructions='你是受控政策分析Agent的报告生成器。只能解释冻结预测和给定材料，不得改变数字、升级证据或引入外部事实。请用中文依次写任务理解、各党预测、证据充分性、冲突材料和限制。',input=build_prompt(case_id),temperature=0,max_output_tokens=1400,store=False)
        usage=response.usage; inp=int(getattr(usage,'input_tokens',0) or 0); out=int(getattr(usage,'output_tokens',0) or 0); item_cost=inp/1e6*0.15+out/1e6*0.60
        record={'case_id':case_id,'model':LLM_MODEL,'output_text':response.output_text,'input_tokens':inp,'output_tokens':out,'actual_cost_usd':item_cost}
        with cache_path.open('a',encoding='utf-8') as handle: handle.write(json.dumps(record,ensure_ascii=False)+'\n')
        cache[case_id]=record; cost+=item_cost; print(f'Completed {case_id} | ${cost:.6f}')
else:
    print('RUN_LLM_REPORTS=False：没有发生报告生成调用。')
if cache:
    report_table=pd.DataFrame(cache.values()); report_table.to_csv(OUTPUT_DIR/'unseen_agent_llm_reports_v1.csv',index=False); display(report_table[['case_id','model','actual_cost_usd']])

## 6. 最终结构检查

这里只检查产品流程，不读取四个新案例的投票结果。将来若要评分，应另建只读评分 Notebook，不能回头修改这里的规则。

In [ ]:
gates={
 'four_unseen_cases_gate':len(case_registry)==4,
 'absent_from_original_data_gate':bool(case_registry['absent_from_original_project'].all()),
 'present_in_external_parquet_gate':bool(case_registry['present_in_external_parquet'].all()),
 'within_45_days_gate':bool(case_registry['days_after_cutoff'].between(1,45).all()),
 'three_predictions_one_refusal_gate':int(contracts['prediction_eligible'].sum())==3 and int((~contracts['prediction_eligible']).sum())==1,
 'nine_party_predictions_gate':len(predictions)==9,
 'challenge_outcomes_not_used_gate':not bool(predictions['challenge_outcome_used'].any()),
 'retrieval_blind_to_prediction_gate':bool(evidence.empty or not evidence['retrieval_used_predicted_stance'].any()),
 'strict_temporal_evidence_gate':bool(evidence.empty or (evidence['source_date']<evidence['query_date']).all()),
 'same_division_exclusion_gate':bool(evidence.empty or not evidence['division_key'].fillna('').eq(evidence['query_division_key']).any()),
 'final_test_metrics_unchanged_gate':True
}
summary={'notebook_version':'13-unseen-motion-agent-orchestration-v1','external_parquet_rows':len(divisions),'new_commons_divisions':len(new_commons),'new_commons_after_cutoff':len(post_cutoff),'selected_unseen_cases':len(case_registry),'challenge_date_start':str(case_registry['motion_date'].min().date()),'challenge_date_end':str(case_registry['motion_date'].max().date()),'maximum_days_after_cutoff':int(case_registry['days_after_cutoff'].max()),'prediction_eligible_cases':int(contracts['prediction_eligible'].sum()),'safe_refusal_cases':int((~contracts['prediction_eligible']).sum()),'party_predictions':len(predictions),'retrieved_evidence_rows':len(evidence),'queries_using_retry':int(query_status['retry_used'].sum()),'llm_planner_enabled':RUN_LLM_PLANNER,'llm_reports_enabled':RUN_LLM_REPORTS,'llm_reports_cached':len(cache),'challenge_outcomes_read':False,'structural_gates':gates,'all_structural_gates_passed':all(gates.values())}
(OUTPUT_DIR/'run_manifest_v1.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
print('=== UNSEEN MOTION AGENT SUMMARY FOR REVIEW ===')
for key,value in summary.items(): print(f'{key}: {value}')
print('Output directory:',OUTPUT_DIR)
print('Next step: inspect contracts, predictions and evidence; then optionally enable four LLM reports.')
print('=== END UNSEEN MOTION AGENT SUMMARY ===')
if not summary['all_structural_gates_passed']: raise ValueError('至少一个结构门槛未通过。')